In [58]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tennie2009ratcheting")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Tennie_2009_Data apes loop study.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [59]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="tennie2009ratcheting"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "ape",
    "formed loop y n":"formed_loop_y_n", 
    "success w loop y n":"success_w_loop_y_n",
    "species":"species_original"}, inplace=True)


In [60]:
df['date'] = df['date of exp'].astype(str).str.pad(6, 'left', '0')
df['day'] = df['date'].str.slice(0,2)
df['month'] = df['date'].str.slice(2,4)
df['year'] = df['date'].str.slice(4,6)
df['year'] = '20' + df['year'].astype(str)

In [61]:
df['ape'].replace(' and ', '&', inplace=True, regex=True)
df[['ape','ape_2']] = df['ape'].str.split('&',expand=True)

In [62]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
df['ape_2'] = df['ape_2'].str.strip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)
    df['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

# df['dyad']=df.ape.str.cat(df.ape_2, sep='_')


In [63]:
# role=[]
# role_2=[]
# for index, row in df.iterrows():
#     if not pd.isna(row['ape']):
#         role.append("subject")
#     else:
#         role.append("")
# df = df.assign(role=role)
# for index, row in df.iterrows(): 
#     if not pd.isna(row['ape_2']):
#         role_2.append("")
#     else:
#         role_2.append("")
# df = df.assign(role_2=role_2)
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"present"}, inplace=True)

In [64]:
df.loc[df.participant == 'viringika', ['present', 'sex_2', 'dyad', 'comments']] = 'kibara', '','','always with kibara'

In [65]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='present', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [66]:
# df['participant'].unique()
df['condition'].replace('bl', 'object_movement', inplace=True, regex=True)
df=df.sort_values(by = ['participant','session','trial'])

In [67]:
tennie2009ratcheting_standardized=df[[ 'study_id',  'year','month', 'day',  'participant', 'age_in_years', 'sex', 
       'present','species','session',
       'trial', 'condition','formed_loop_y_n', 'success_w_loop_y_n',
       'comments']]
comp_out_path_stand = os.path.join(out_pathway, 'tennie2009ratcheting_standardized.csv')
tennie2009ratcheting_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =tennie2009ratcheting_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
tennie2009ratcheting_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tennie2009ratcheting_glossary.csv')
tennie2009ratcheting_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

